# 2. Hand-made feature vectors for words and expressions

Before learning vectors with neural networks, we build them **by hand**, with simple algorithms.
Each word or expression (an **n-gram**) found in the reports of Notebook 1 receives a vector of
four positions:

| Position | Meaning | How it is computed |
|---|---|---|
| 1 | the n-gram is a **disease** | lookup in the disease dictionary |
| 2 | the n-gram is a **symptom** | lookup in the symptom dictionary |
| 3 | the n-gram is a **negation word** | lookup in the negation cue list |
| 4 | the n-gram is **in negation scope** | a negation cue appears earlier in the same sentence, with no terminator ("but", "however") in between |

*Metaphor:* the lexicon is a **field guide**. For each creature (expression) it tells you what kind of
creature it is. Position 4 is not in the guide: it depends on **where** the creature was spotted.

Run this notebook after `01_generate_reports.ipynb`.

In [1]:
import re
from collections import defaultdict

import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SENTENCES_FILE = "sentences.csv"
ANNOTATIONS_FILE = "annotations.csv"
LEXICON_FILE = "lexicon.csv"
VECTORS_FILE = "vectors.csv"

# Longest n-gram considered ("trembling on the little finger" has 5 tokens).
MAX_N = 5

FEATURES = ["f1_disease", "f2_symptom", "f3_negation", "f4_neg_scope"]

## Load the sentences and build the dictionaries

Each dictionary maps a **surface form** (as written in the text) to its **canonical** name.

In [2]:
sentences = pd.read_csv(SENTENCES_FILE)
annotations = pd.read_csv(ANNOTATIONS_FILE)
lexicon = pd.read_csv(LEXICON_FILE)

def dictionary(kind):
    rows = lexicon[lexicon["type"] == kind]
    return dict(zip(rows["surface"], rows["canonical"]))

DISEASE_DICT = dictionary("disease")
SYMPTOM_DICT = dictionary("symptom")
NEGATION_CUES = set(dictionary("negation"))
TERMINATORS = set(dictionary("terminator"))

print(f"{len(sentences)} sentences")
print(f"{len(DISEASE_DICT)} disease forms, {len(SYMPTOM_DICT)} symptom forms")
print(f"negation cues: {sorted(NEGATION_CUES)}")
print(f"terminators:   {sorted(TERMINATORS)}")

122 sentences
12 disease forms, 18 symptom forms
negation cues: ['absence', 'absent', 'denies', 'free of', 'negative', 'no', 'not', 'ruled out', 'without']
terminators:   ['but', 'however']


## Tokenize and extract n-grams

Tokens are lowercase alphanumeric sequences; punctuation is discarded. An n-gram is a sequence of
`n` consecutive tokens, identified by its start position.

In [3]:
def tokenize(sentence):
    return re.findall(r"[a-z0-9]+", sentence.lower())

def extract_ngrams(tokens, n):
    return [(i, " ".join(tokens[i:i + n])) for i in range(len(tokens) - n + 1)]

example = sentences["sentence"].iloc[0]
tokens = tokenize(example)
print(example)
print(tokens)
for n in (1, 2, 3):
    print(n, extract_ngrams(tokens, n))

The patient presents with paralysis.
['the', 'patient', 'presents', 'with', 'paralysis']
1 [(0, 'the'), (1, 'patient'), (2, 'presents'), (3, 'with'), (4, 'paralysis')]
2 [(0, 'the patient'), (1, 'patient presents'), (2, 'presents with'), (3, 'with paralysis')]
3 [(0, 'the patient presents'), (1, 'patient presents with'), (2, 'presents with paralysis')]


## Negation scope

A negation cue can have more than one token ("free of", "ruled out"). For every token position we
mark whether it is **inside the scope** of a cue: it comes after the end of a cue and before the end of
the sentence or a terminator. This is a tiny version of the NegEx algorithm.

In [4]:
def cue_spans(tokens):
    """(start, end) of each negation cue; end is exclusive."""
    spans = []
    for n in range(1, MAX_N + 1):
        for start, gram in extract_ngrams(tokens, n):
            if gram in NEGATION_CUES:
                spans.append((start, start + n))
    return spans

def negation_scope(tokens):
    """in_scope[i] is True if token i is governed by a preceding negation cue."""
    in_scope = [False] * len(tokens)
    for _, end in cue_spans(tokens):
        for i in range(end, len(tokens)):
            if tokens[i] in TERMINATORS:
                break
            in_scope[i] = True
    return in_scope

for s in ["No signs of paralysis.", "The patient is free of chest pain.",
          "Examination reveals yellowish tongue.", "Signs of paralysis were not observed."]:
    t = tokenize(s)
    print(f"{s:40} {[w for w, inside in zip(t, negation_scope(t)) if inside]}")

No signs of paralysis.                   ['signs', 'of', 'paralysis']
The patient is free of chest pain.       ['chest', 'pain']
Examination reveals yellowish tongue.    []
Signs of paralysis were not observed.    ['observed']


The last example is the **hard mode** case: the cue comes *after* the term, so the term is not in
scope. Keep it in mind for the evaluation below.

## The feature vector of each n-gram occurrence

In [5]:
def vectorize(sentence):
    tokens = tokenize(sentence)
    in_scope = negation_scope(tokens)
    rows = []
    for n in range(1, MAX_N + 1):
        for start, gram in extract_ngrams(tokens, n):
            rows.append({
                "start": start,
                "n": n,
                "ngram": gram,
                "f1_disease": int(gram in DISEASE_DICT),
                "f2_symptom": int(gram in SYMPTOM_DICT),
                "f3_negation": int(gram in NEGATION_CUES),
                "f4_neg_scope": int(in_scope[start]),
            })
    return rows

pd.DataFrame(vectorize("The patient is free of chest pain.")).query("n <= 2")

,start,n,ngram,f1_disease,f2_symptom,f3_negation,f4_neg_scope
0,0,1,the,0,0,0,0
1,1,1,patient,0,0,0,0
2,2,1,is,0,0,0,0
3,3,1,free,0,0,0,0
4,4,1,of,0,0,0,0
5,5,1,chest,0,0,0,1
6,6,1,pain,0,0,0,1
7,0,2,the patient,0,0,0,0
8,1,2,patient is,0,0,0,0
9,2,2,is free,0,0,0,0


In [6]:
vector_rows = []
for _, s in sentences.iterrows():
    for row in vectorize(s["sentence"]):
        vector_rows.append({"report_id": s["report_id"], "sentence_id": s["sentence_id"],
                            "section": s["section"], **row})

vectors = pd.DataFrame(vector_rows)
vectors.to_csv(VECTORS_FILE, index=False)

print(f"{len(vectors)} n-gram occurrences saved to {VECTORS_FILE}")
vectors.groupby(FEATURES).size().rename("occurrences").reset_index()

2384 n-gram occurrences saved to vectors.csv


,f1_disease,f2_symptom,f3_negation,f4_neg_scope,occurrences
0,0,0,0,0,1781
1,0,0,0,1,450
2,0,0,1,0,56
3,0,1,0,0,31
4,0,1,0,1,26
5,1,0,0,0,25
6,1,0,0,1,15


Most occurrences have the vector `[0, 0, 0, 0]` or `[0, 0, 0, 1]`: they are words or pieces of
expressions that the field guide does not know. Only a few n-grams "light up" positions 1–3.

## Longest match

When two dictionary hits overlap, we keep the **longest** one. In the current lexicon no entry is
nested inside another, so this step changes nothing yet. It becomes essential as soon as we add a
shorter entry such as "infection", which would otherwise also be found inside "bacterial infection".

In [7]:
def longest_matches(hits):
    """Keep dictionary hits that are not contained in a longer hit of the same sentence."""
    kept = []
    for sentence_id, group in hits.groupby("sentence_id"):
        spans = sorted(group.itertuples(), key=lambda r: -r.n)
        taken = set()
        for r in spans:
            positions = set(range(r.start, r.start + r.n))
            if not positions & taken:
                taken |= positions
                kept.append(r.Index)
    return hits.loc[sorted(kept)]

hits = vectors[(vectors["f1_disease"] == 1) | (vectors["f2_symptom"] == 1)]
terms = longest_matches(hits)
print(f"{len(hits)} dictionary hits, {len(terms)} after longest match")

97 dictionary hits, 97 after longest match


## From vectors back to facts

Two rules are enough to read the reports:

* `f2 = 1` and `f4 = 0` → the symptom is **present** (`f1` for diseases);
* `f2 = 1` and `f4 = 1` → the symptom is **absent**.

We compare the extracted facts with the ground truth saved by Notebook 1.

In [8]:
extracted = terms.assign(
    canonical=[DISEASE_DICT.get(g) or SYMPTOM_DICT.get(g) for g in terms["ngram"]],
    type=["disease" if d else "symptom" for d in terms["f1_disease"]],
    predicted=["absent" if f4 else "present" for f4 in terms["f4_neg_scope"]],
)[["sentence_id", "ngram", "canonical", "type", "predicted"]]

truth = annotations[annotations["type"].isin(["symptom", "disease"])]
evaluation = truth.merge(extracted, on=["sentence_id", "canonical", "type"], how="outer")
evaluation["predicted"] = evaluation["predicted"].fillna("missed")
evaluation["polarity"] = evaluation["polarity"].fillna("spurious")

accuracy = (evaluation["polarity"] == evaluation["predicted"]).mean()
print(f"Accuracy: {accuracy:.1%} over {len(evaluation)} facts")
pd.crosstab(evaluation["polarity"], evaluation["predicted"], rownames=["truth"], colnames=["predicted"])

Accuracy: 100.0% over 97 facts


predicted,absent,present
truth,,
absent,41,0
present,0,56


In [9]:
# The sentences where the simple rules fail (empty unless hard mode is on).
errors = evaluation[evaluation["polarity"] != evaluation["predicted"]]
errors.merge(sentences, on="sentence_id")[["sentence", "canonical", "polarity", "predicted"]]

,sentence,canonical,polarity,predicted


**Try it:** set `POST_NEGATION = True` in Notebook 1, run it again, and then run this notebook.
Sentences such as "Signs of paralysis were not observed." break the rule, because position 4 only
looks **backward**.

## From occurrences to expression vectors: a "passport"

So far each **occurrence** has a vector. Averaging the vectors of all occurrences of the same
expression gives one vector per expression, like a passport. Positions 1–3 stay 0 or 1, because they
come from the field guide. Position 4 becomes a **negation rate**: how often the expression was seen
inside a negation.

This table (one row per expression, one column per property) is our first **embedding-like**
representation.

In [10]:
known = vectors[(vectors[["f1_disease", "f2_symptom", "f3_negation"]].sum(axis=1) > 0)]

passports = (
    known.groupby("ngram")[FEATURES].mean()
    .join(known.groupby("ngram").size().rename("count"))
    .sort_values(["f1_disease", "f2_symptom", "f3_negation", "count"], ascending=False)
    .round(2)
)
passports

,f1_disease,f2_symptom,f3_negation,f4_neg_scope,count
ngram,,,,,
bacterial infection,1.0,0.0,0.0,0.57,7
infection of bacterial origin,1.0,0.0,0.0,0.00,6
bite deficit,1.0,0.0,0.0,0.80,5
deficit caused by bite,1.0,0.0,0.0,0.20,5
bacterial infectious process,1.0,0.0,0.0,0.25,4
altercation injury,1.0,0.0,0.0,0.33,3
infection of viral origin,1.0,0.0,0.0,0.67,3
bite related deficit,1.0,0.0,0.0,0.00,2
fight injury,1.0,0.0,0.0,0.50,2


In [11]:
# The same idea per canonical term: all synonyms share one passport.
by_canonical = known.assign(
    canonical=[DISEASE_DICT.get(g) or SYMPTOM_DICT.get(g) or g for g in known["ngram"]]
).groupby("canonical")

canonical_passports = (
    by_canonical[FEATURES].mean()
    .join(by_canonical.size().rename("count"))
    .sort_values(["f1_disease", "f2_symptom", "count"], ascending=False)
    .round(2)
)
canonical_passports

,f1_disease,f2_symptom,f3_negation,f4_neg_scope,count
canonical,,,,,
bacterial infection,1.0,0.0,0.0,0.29,17
bite deficit,1.0,0.0,0.0,0.42,12
fights,1.0,0.0,0.0,0.50,6
viral infection,1.0,0.0,0.0,0.40,5
yellow tongue,0.0,1.0,0.0,0.09,11
paralysis,0.0,1.0,0.0,0.80,10
trembling on the little finger,0.0,1.0,0.0,0.40,10
member loss,0.0,1.0,0.0,0.33,9
severe anger,0.0,1.0,0.0,0.78,9


## Where to go next

* **More positions** (a triage form with more checkboxes): pathogen dictionary, hedge cues
  ("possible", "suspected"), history cues, section of the report, distance to the nearest cue.
* **Weighting:** TF-IDF, as in `information-retrieval/basics/cabot/ir-03-tfidf.ipynb`.
* **Company it keeps:** a symptom × disease co-occurrence matrix, compared with cosine similarity.
* **A map:** PPMI + SVD in 2 dimensions, the bridge to dense embeddings.
* **Learned embeddings:** a tiny word2vec on the generated sentences, compared with these passports.